# 🧪 Step 2: Evaluate RAG System using RAGAS
This notebook loads the generated testset CSV (`eval_results/testset.csv`), queries your running RAG API to collect search contexts and answers, and evaluates the system's performance using RAGAS metrics.

In [ ]:
!pip install ragas langchain-openai python-dotenv requests datasets pandas tqdm

## ⚙️ Configuration
Edit `PROJECT_ID` and `BASE_URL` to match your running FastAPI instance.

In [4]:
import os
import pandas as pd
from dotenv import load_dotenv

# Load .env from src/.env (go up one level from notebooks/)
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), 'src', '.env'))

# ── Edit these ──────────────────────────────────────────────
PROJECT_ID = 40
BASE_URL = "http://localhost:5000"
INPUT_PATH = "eval_results/testset.csv"
OUTPUT_PATH = "eval_results/results.csv"
# ────────────────────────────────────────────────────────────

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in src/.env!"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in src/.env!"

print(f"✅ Config loaded. Project ID: {PROJECT_ID}")

✅ Config loaded. Project ID: 40


## 📥 Step 1: Load Testset
We load the Q&A dataset generated in Step 1.

In [5]:
df_testset = pd.read_csv(INPUT_PATH)
print(f"✅ Loaded {len(df_testset)} test cases.")
df_testset.head(3)

✅ Loaded 54 test cases.


,user_input,reference,reference_contexts
0,What types of data sources are covered in the ...,The learning material covers working with data...,explains the solution and provides meaningful ...
1,How long has O'Reilly Media been providing tec...,O'Reilly Media has provided technology and bus...,explains the solution and provides meaningful ...
2,When was the second edition of the book released?,The second edition of the book was released in...,"Preface\nxiii Of course, the machine learning ..."


## ⚙️ Step 2: Query RAG Backend to Collect Output
For each question, we query the FastAPI backend:
1. `POST /api/v1/nlp/index/search/{project_id}` to get retrieved chunks (context)
2. `POST /api/v1/nlp/index/answer/{project_id}` to get the generated answer

In [6]:
import requests
from tqdm.auto import tqdm

eval_data = []

for idx, row in tqdm(df_testset.iterrows(), total=len(df_testset), desc="Querying RAG system"):
    question = row["user_input"]
    reference = row["reference"]
    
    # 1. Retrieve contexts from the search endpoint
    search_resp = requests.post(
        f"{BASE_URL}/api/v1/nlp/index/search/{PROJECT_ID}",
        json={"text": question, "limit": 5}
    )
    if search_resp.status_code != 200:
        print(f"⚠️ Search failed for: '{question[:30]}...' -> HTTP {search_resp.status_code}")
        continue
    
    search_results = search_resp.json().get("results", [])
    contexts = [r.get("text", "") for r in search_results]
    
    # 2. Get RAG answer from the answer endpoint
    answer_resp = requests.post(
        f"{BASE_URL}/api/v1/nlp/index/answer/{PROJECT_ID}",
        json={"text": question, "limit": 5}
    )
    if answer_resp.status_code != 200:
        print(f"⚠️ Answer generation failed for: '{question[:30]}...' -> HTTP {answer_resp.status_code}")
        continue
        
    answer = answer_resp.json().get("answer", "")
    
    eval_data.append({
        "question": question,
        "contexts": contexts,
        "answer": answer,
        "ground_truth": reference
    })

print(f"✅ Collected RAG responses for {len(eval_data)} questions.")

Querying RAG system: 100%|██████████| 54/54 [02:24<00:00,  2.67s/it]

✅ Collected RAG responses for 54 questions.


## 📊 Step 3: Run RAGAS Evaluation
We wrap our OpenAI model using Ragas wrappers and run the evaluation on the collected dataset.

In [8]:
from langchain_openai import ChatOpenAI

# Add your Groq API keys here (create free accounts at console.groq.com)
GROQ_API_KEYS = [
    "gsk_REDACTED_KEY",
    "gsk_REDACTED_KEY",
    "gsk_REDACTED_KEY",
    "gsk_REDACTED_KEY",
    "gsk_REDACTED_KEY",
    "gsk_REDACTED_KEY",
    "gsk_REDACTED_KEY",
    # Add as many as you want
]

# Pre-create one client per key
_clients = [
    ChatOpenAI(
        model="llama-3.3-70b-versatile",
        api_key=key,
        base_url="https://api.groq.com/openai/v1",
    )
    for key in GROQ_API_KEYS
]
_state = {"idx": 0}

class RotatingGroqLLM(ChatOpenAI):
    """Auto-rotates to the next API key on rate limit errors."""

    async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs):
        kwargs["n"] = 1
        tried = 0
        while tried < len(_clients):
            try:
                return await _clients[_state["idx"]]._agenerate(
                    messages, stop=stop, run_manager=run_manager, **kwargs
                )
            except Exception as e:
                if "429" in str(e) or "rate_limit" in str(e).lower():
                    old = _state["idx"] + 1
                    _state["idx"] = (_state["idx"] + 1) % len(_clients)
                    print(f"🔄 Key #{old} rate limited → switching to key #{_state['idx']+1}/{len(_clients)}")
                    tried += 1
                else:
                    raise
        raise Exception("❌ All API keys exhausted!")

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        kwargs["n"] = 1
        tried = 0
        while tried < len(_clients):
            try:
                return _clients[_state["idx"]]._generate(
                    messages, stop=stop, run_manager=run_manager, **kwargs
                )
            except Exception as e:
                if "429" in str(e) or "rate_limit" in str(e).lower():
                    old = _state["idx"] + 1
                    _state["idx"] = (_state["idx"] + 1) % len(_clients)
                    print(f"🔄 Key #{old} rate limited → switching to key #{_state['idx']+1}/{len(_clients)}")
                    tried += 1
                else:
                    raise
        raise Exception("❌ All API keys exhausted!")

print(f"✅ {len(GROQ_API_KEYS)} Groq API keys loaded and ready to rotate")

✅ 7 Groq API keys loaded and ready to rotate


In [9]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

dataset = Dataset.from_dict({
    "question": [x["question"] for x in eval_data],
    "contexts": [x["contexts"] for x in eval_data],
    "answer": [x["answer"] for x in eval_data],
    "ground_truth": [x["ground_truth"] for x in eval_data]
})

# Uses the rotating key pool defined above
lc_llm = RotatingGroqLLM(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEYS[0],
    base_url="https://api.groq.com/openai/v1",
)
lc_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY, chunk_size=8)

evaluator_llm = LangchainLLMWrapper(lc_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)

print("🚀 Running RAGAS evaluation...")
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
)

print("\n✅ Evaluation Completed!")
print(result)

/tmp/ipykernel_218445/3914410752.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_218445/3914410752.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_218445/3914410752.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_218445/3914410752.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be

🚀 Running RAGAS evaluation...


Evaluating:   0%|          | 1/216 [00:02<09:22,  2.62s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   5%|▍         | 10/216 [00:12<03:42,  1.08s/it]

🔄 Key #1 rate limited → switching to key #2/7


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   5%|▌         | 11/216 [00:17<05:49,  1.70s/it]

...

🔄 Key #1 rate limited → switching to key #2/7
🔄 Key #2 rate limited → switching to key #3/7
🔄 Key #3 rate limited → switching to key #4/7
🔄 Key #4 rate limited → switching to key #5/7


Exception raised in Job[182]: Exception(❌ All API keys exhausted!)
Evaluating:  99%|█████████▉| 214/216 [07:46<00:01,  1.03it/s]

🔄 Key #5 rate limited → switching to key #6/7
🔄 Key #6 rate limited → switching to key #7/7


Evaluating: 100%|██████████| 216/216 [07:47<00:00,  2.16s/it]


🔄 Key #7 rate limited → switching to key #1/7
🔄 Key #1 rate limited → switching to key #2/7

✅ Evaluation Completed!
{'faithfulness': 0.8936, 'answer_relevancy': 0.8387, 'context_precision': 0.8758, 'context_recall': 0.9286}


## 💾 Step 4: Save and Display Results
We convert the evaluation results into a pandas DataFrame, format them nicely, and save them to a CSV file.

In [11]:
import os
import pandas as pd

df_results = result.to_pandas()

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_results.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Results saved to {OUTPUT_PATH}")

# Show actual column names first
print(f"Columns: {list(df_results.columns)}")

pd.set_option('display.max_colwidth', 100)
df_results[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]]

✅ Results saved to eval_results/results.csv
Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,What types of data sources are covered in the provided learning material?,1.000000,0.783129,0.333333,1.0
1,How long has O'Reilly Media been providing technology and business training?,1.000000,0.984417,1.000000,1.0
2,When was the second edition of the book released?,1.000000,0.957246,1.000000,1.0
3,"What is the purpose of using pretrained classifiers in machine learning pipelines, as introduced...",1.000000,0.842709,1.000000,1.0
4,What are the two preprocessing steps contained in the preprocess object?,1.000000,0.950121,1.000000,1.0
5,"What type of machine learning is discussed in much of the book, where access to both features an...",1.000000,0.533176,0.805556,1.0
6,What is the purpose of the chi-square (χ²) statistic in the context of this book?,1.000000,0.689458,0.679167,1.0
7,Why was the second edition of this book easier to create for the first author?,1.000000,0.957273,1.000000,1.0
8,"What typographical convention is used in the book to indicate new terms, URLs, email addresses, ...",0.500000,0.469461,1.000000,1.0
9,How is the issue of imbalanced classes handled in the context of categorical data?,1.000000,0.000000,0.000000,0.0


# 📊 RAG Evaluation Metrics (RAGAS)

To evaluate the performance of our Retrieval-Augmented Generation (RAG) system, we use four key metrics from the **RAGAS** framework. These metrics assess the two main components of the pipeline: **Retrieval** (finding the right information) and **Generation** (writing the correct answer).

---

### 1. Faithfulness
* **Focus Area:** Generator (LLM) Accuracy
* **What it measures:** Factuality and hallucination control.
* **Simple Definition:** Does the generated answer only contain facts that exist in the retrieved documents? 
* **How it works:** It breaks down the generated answer into individual statements and verifies if each statement can be directly inferred from the retrieved contexts. 
* **Why it matters:** A high score (close to **1.0**) means the LLM is not making things up (hallucinating) and is strictly sticking to the provided documents.

---

### 2. Answer Relevancy
* **Focus Area:** Generator (LLM) Quality
* **What it measures:** How directly the generated answer addresses the user's question.
* **Simple Definition:** Does the generated answer actually answer the user's question, or does it go off-topic?
* **How it works:** It analyzes the generated answer, prompts an LLM to guess what question this answer is responding to, and measures how close that guessed question is to the user's actual question.
* **Why it matters:** An answer can be 100% faithful to the text (not hallucinated), but still completely fail to address what the user actually asked. High relevancy means the answer is concise and on-point.

---

### 3. Context Precision
* **Focus Area:** Retrieval System Quality
* **What it measures:** Search ranking quality.
* **Simple Definition:** Are the most relevant chunks of information ranked at the very top of the retrieved results?
* **How it works:** It evaluates whether the retrieved text chunks containing the information needed to answer the question are placed at the beginning of the context window.
* **Why it matters:** LLMs pay more attention to information at the beginning of a prompt. High precision ensures that the most important context isn't buried deep in the text, which prevents the LLM from missing it.

---

### 4. Context Recall
* **Focus Area:** Retrieval System Quality
* **What it measures:** Search completeness.
* **Simple Definition:** Did the search system retrieve *all* the necessary information required to construct the ground-truth answer?
* **How it works:** It compares the ground-truth reference answer against the retrieved context to see if every key point in the ground truth is present in the retrieved texts.
* **Why it matters:** If this score is low, it means the retriever missed crucial parts of the document, making it impossible for the LLM to write a complete and correct answer.

---

### 💡 Quick Reference Summary Table

| Metric | Component Evaluated | Key Inputs Used | What High Score Means |
| :--- | :--- | :--- | :--- |
| **Faithfulness** | Generator (LLM) | Generated Answer + Contexts | **No Hallucinations:** The answer is fully grounded in the retrieved text. |
| **Answer Relevancy** | Generator (LLM) | Question + Generated Answer | **On-Topic:** The answer is direct, concise, and avoids fluff. |
| **Context Precision** | Retriever (Search) | Question + Contexts | **Good Ranking:** The most useful search results are at the top. |
| **Context Recall** | Retriever (Search) | Ground Truth + Contexts | **Complete Search:** The system retrieved all the necessary facts. |

In [12]:
# Calculate the mean for each metric
averages = df_results[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].mean()

# Convert it to a nice DataFrame for the report
df_averages = averages.to_frame(name="Average Score").reset_index()
df_averages.rename(columns={"index": "Metric"}, inplace=True)

# Format the scores to 4 decimal places for a cleaner look
df_averages["Average Score"] = df_averages["Average Score"].round(4)

# Display the table
print("📊 Evaluation Averages for Report:")
display(df_averages)

# Optionally, save this small table to a CSV as well!
df_averages.to_csv("eval_results/averages.csv", index=False)

📊 Evaluation Averages for Report:


,Metric,Average Score
0,faithfulness,0.8936
1,answer_relevancy,0.8387
2,context_precision,0.8758
3,context_recall,0.9286
